## Dataset loader

In [1]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return dataloader


with open("text_sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

vocab_size = 50257
output_dim = 256
max_len = 1024
context_length = max_len


token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 4

dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)

context_length = max_length
d_in = output_dim

In [2]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

print(input_embeddings.shape)

torch.Size([8, 4, 256])


## Basic Self Attention

In [3]:
class SelfAttention(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_weights = torch.softmax(attn_scores/ keys.shape[-1]**0.5, dim=-1)

        context_vector = attn_weights @ values
        return context_vector

torch.manual_seed(789)
d_in = output_dim
d_out = output_dim
sa = SelfAttention(d_in, d_out)
# print(sa(input_embeddings))


## Casual Self Attention

In [28]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal = 1))

    def forward(self, x):
        batch, n_tokens, _ = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_score = queries @ keys.transpose(1,2)
        attn_score.masked_fill_(self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)

        attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vector = attn_weights @ values

        return context_vector


torch.manual_seed(789)
d_in = output_dim
d_out = output_dim
sa = CausalAttention(d_in, d_out, context_length=context_length, dropout=0.2)
# print(sa(input_embeddings))

## Multihead Self Attention

In [40]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.d_in = d_in
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = self.d_out // num_heads
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)


    def forward(self, x):
        batch, num_tokens, dim = x.shape

        queries = self.W_query(x)
        values = self.W_value(x)
        keys = self.W_key(x)

        queries = queries.view(batch, num_tokens, self.num_heads, self.head_dim)
        values = values.view(batch, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(batch, num_tokens, self.num_heads, self.head_dim)

        queries = queries.transpose(1,2)
        values = values.transpose(1,2)
        keys = keys.transpose(1,2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[ :num_tokens, :num_tokens], -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_weights @ values).transpose(1,2)
        context_vector = context_vector.contiguous().view(batch, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)

        return context_vector



torch.manual_seed(789)
d_in = output_dim
d_out = output_dim
sa = MultiHeadAttention(d_in, d_out, context_length = context_length, num_heads = 2, dropout = 0.5)
# print(sa(input_embeddings))

## Multihead Attention with combined weights

In [6]:
import torch.nn as nn

class MultiHeadAttentionCombinedQKV(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0,  "d_out is indivisible by num_heads"
        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x) # last dimension: embed_dim * 3
        
        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        queries, keys, values = qkv.unbind(0)

        # (b, num_heads, num_tokens, head_dim) --> (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2,3)
        attn_scores = attn_scores.masked_fill(
            self.mask.bool()[:num_tokens, :num_tokens],
            -torch.inf
        )

        attn_weights = torch.softmax(attn_scores/ keys.shape[-1], dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # (b, num_heads, num_tokens, num_tokens) --> (b, num_heads, num_tokens, head_dim)
        context_vector = attn_weights @ values
        # (b, num_heads, num_tokens, head_dim) --> (b, num_tokens, num_heads, head_dim)
        context_vector = context_vector.transpose(1,2)

        # (b, num_tokens, num_heads, head_dim) --> (b, num_tokens, embed_dim)
        context_vector = context_vector.contiguous().view(batch_size, num_tokens, self.num_heads * self.head_dim)
        context_vector = self.proj(context_vector)

        return context_vector


batch_size = 8
context_len = 1024
embed_dim = 768
embeddings = torch.randn((batch_size, context_len, embed_dim))


mha_combined_qkv = MultiHeadAttentionCombinedQKV(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
)
out = mha_combined_qkv(embeddings)

## Multihead attention via Einsum

In [7]:
import math


class MHAEinsum(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Parameter(torch.randn(d_in, d_out))
        self.W_key = nn.Parameter(torch.randn(d_in, d_out))
        self.W_value = nn.Parameter(torch.randn(d_in, d_out))
        
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.W_query, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W_key, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W_value, a=math.sqrt(5))

    def forward(self, x):
        b, n, _ = x.shape

        Q = torch.einsum("bnd, do->bno", x, self.W_query)
        K = torch.einsum("bnd, do->bno", x, self.W_key)
        V = torch.einsum("bnd, do->bno", x, self.W_value)

        Q = Q.view(b, n, self.num_heads, self.head_dim).transpose(1,2)
        V = V.view(b, n, self.num_heads, self.head_dim).transpose(1,2)
        K = K.view(b, n, self.num_heads, self.head_dim).transpose(1,2)

        attn_score = torch.einsum("bhnd,bhmd->bhnm", Q, K) / (self.head_dim ** 0.5)

        mask = self.mask[:n, :n]
        attn_score = attn_score.masked_fill(mask.bool(), -torch.inf)

        attn_weights = torch.softmax(attn_score, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = torch.einsum("bhnm,bhmd->bhnd", attn_weights, V)
        context_vec = context_vec.transpose(1, 2).reshape(b, n, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec
    

mha_einsum = MHAEinsum(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
)

out = mha_einsum(embeddings)
print(out.shape)

torch.Size([8, 1024, 768])


## Grouped-Query Attention (GQA)

In [8]:
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, d_out, context_length, dropout, num_heads, num_kv_groups, dtype=None, qkv_bias=False, max_seq_len=None, window_size=None
    ):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_key = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=qkv_bias, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=qkv_bias, dtype=dtype)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias, dtype=dtype)

        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype)
        self.dropout = nn.Dropout(dropout)

        self.max_seq_len = max_seq_len or context_length
        self.window_size = window_size or self.max_seq_len

        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)


    def forward(self, x, use_cache=True):
        batch, num_tokens, dim = x.shape
        
        if use_cache:
            assert num_tokens <= self.window_size, (
                f"Input chunk size ({num_tokens}) exceeds KV cache window size ({self.window_size}). "
            )

        queries = self.W_query(x)
        values_new = self.W_value(x)
        keys_new = self.W_key(x)

        queries = queries.view(batch, num_tokens, self.num_heads, self.head_dim)
        values_new = values_new.view(batch, num_tokens, self.num_kv_groups, self.head_dim)
        keys_new = keys_new.view(batch, num_tokens, self.num_kv_groups, self.head_dim)

        queries = queries.transpose(1,2)
        values_new = values_new.transpose(1,2)
        keys_new = keys_new.transpose(1,2)

        if use_cache:

            if self.cache_k is None or self.cache_k.size(0) != batch:
                self.cache_k = torch.zeros(batch, self.num_kv_groups, self.window_size, self.head_dim,  device=x.device)
                self.cache_v = torch.zeros(batch, self.num_kv_groups, self.window_size, self.head_dim,  device=x.device)
                self.ptr_cur = 0

            if self.ptr_cur + num_tokens > self.window_size:
                overflow = self.ptr_cur + num_tokens  - self.window_size
                self.cache_k[:, :, :-overflow, :] = self.cache_k[:, :, overflow:, :].clone()
                self.cache_v[:, :, :-overflow, :] = self.cache_v[:, :, overflow:, :].clone()
                self.ptr_cur -= overflow


            self.cache_k[:, :,self.ptr_cur:self.ptr_cur + num_tokens, :] = keys_new
            self.cache_v[:, :,self.ptr_cur:self.ptr_cur + num_tokens, :] = values_new
            self.ptr_cur += num_tokens

            keys = self.cache_k[:, :, :self.ptr_cur, :]
            values = self.cache_v[:, :, :self.ptr_cur, : ]

        else:
            keys, values = keys_new, values_new
            self.ptr_cur = 0

        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)
        attn_scores = queries @ keys.transpose(2, 3)
        K = attn_scores.size(-1) # number of Keys
        if num_tokens == K:
            # No cache → use the pre‑baked triangular mask slice
            casual_mask = torch.triu(torch.ones(num_tokens, K, device=x.device, dtype=torch.bool), diagonal=1)

        else:
            offset = K - num_tokens
            row_idx = torch.arange(num_tokens, device=x.device).unsqueeze(1)
            col_idx = torch.arange(K, device=x.device).unsqueeze(0)
            casual_mask = row_idx + offset < col_idx


        attn_scores.masked_fill_(casual_mask.unsqueeze(0).unsqueeze(0), -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_weights @ values).transpose(1,2)
        context_vector = context_vector.contiguous().view(batch, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)

        return context_vector
    


group_attn = GroupedQueryAttention(
    d_in=768,
    d_out=768,
    context_length=context_len, dropout=0.1, num_heads=12, num_kv_groups=2, qkv_bias=False, max_seq_len=context_len, window_size=context_len
)

out = group_attn(embeddings)
print(out.shape)

torch.Size([8, 1024, 768])


## Multi-Head Latent Attention (MLA)

In [9]:
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_in, d_out, dropout, num_heads, qkv_bias, latent_dim=None):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.latent_dim = latent_dim if latent_dim is not None else max(16, d_out // 8)

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_DKV = nn.Linear(d_in, latent_dim, bias=qkv_bias)
        self.W_UK = nn.Linear(latent_dim, d_out, bias=qkv_bias)
        self.W_UV = nn.Linear(latent_dim, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("cache_c_kv", None, persistent=False)
        self.ptr_current_pos = 0

    def reset_cache(self):
        self.cache_c_kv = None
        self.ptr_current_pos = 0

    @staticmethod
    def _reshape_to_heads(x, num_heads, head_dim):
        batch, num_tokens, _ = x.shape
        return x.view(batch, num_tokens, num_heads, head_dim).transpose(1,2).contiguous()


    def forward(self, x, use_cache=False):
        b, num_tokens, _ = x.shape
        num_heads = self.num_heads
        head_dim = self.head_dim

        queries_all = self.W_query(x)
        latent_new = self.W_DKV(x)

        if use_cache:
            if self.cache_c_kv is None:
                latent_total = latent_new
            else:
                latent_total = torch.cat([self.cache_c_kv, latent_new], dim=1)
            self.cache_c_kv = latent_total
        else:
            latent_total = latent_new

        keys_all = self.W_UK(latent_total)
        values_all = self.W_UV(latent_total)

        queries = self._reshape_to_heads(queries_all, num_heads, head_dim)
        keys = self._reshape_to_heads(keys_all, num_heads, head_dim)
        values = self._reshape_to_heads(values_all, num_heads, head_dim)


        attn_scores = queries @ keys.transpose(2, 3)
        num_tokens_Q = queries.shape[-2]
        num_tokens_k = keys.shape[-2]

        device = queries.device
        if use_cache:
            q_positions = torch.arange(
                self.ptr_current_pos,
                self.ptr_current_pos + num_tokens_Q,
                device=device,
                dtype=torch.long
            )
            self.ptr_current_pos += num_tokens_Q
        else:
            q_positions = torch.arange(num_tokens_Q, device=device, dtype=torch.long)

        k_positions = torch.arange(num_tokens_k, device=device, dtype=torch.long)
        mask_bool = q_positions.unsqueeze(-1) < k_positions.unsqueeze(0)

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

    
mla_attn = MultiHeadLatentAttention(
    d_in=768,
    latent_dim = 128,
    d_out=768,
    dropout=0.1, num_heads=12, qkv_bias=False
)

out = mla_attn(embeddings)
print(out.shape)

torch.Size([8, 1024, 768])


## Sliding Window Attention (SWA)

In [10]:
class MultiHeadAttentionWithSWA(nn.Module):
    def __init__(self, d_in, d_out, dropout, num_heads, qkv_bias=False, sliding_window_size=None):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.sliding_window_size = sliding_window_size

        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.ptr_current_pos = 0

    
    def forward(self, x, use_cache=True):
        b, num_tokens, d_in = x.shape
        
        keys_new = self.W_key(x)
        values_new = self.W_value(x)
        queries = self.W_query(x)

        keys_new = keys_new.view(b, num_tokens, self.num_heads, self.head_dim)
        values_new = values_new.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        if use_cache:
            old_cache_k, old_cache_v = self.cache_k, self.cache_v
            old_len = 0 if old_cache_k is None else old_cache_k.size(1)
            if old_cache_k is None:
                combined_v, combined_k = values_new, keys_new
            else:
                combined_v = torch.cat([old_cache_v, values_new], dim=1)
                combined_k = torch.cat([old_cache_k, keys_new], dim=1)

            keys, values = combined_k, combined_v
            if self.sliding_window_size is not None:
                # Sliding-window attention during chunked prefill:
                #
                # We process multiple query tokens together (a "chunk"), but each query token
                # must still behave exactly like autoregressive decoding.
                #
                # Example:
                #
                #   sliding_window_size = 5
                #   past cache          = [1 2 3 4 5]
                #   current chunk       = [A B C]
                #
                # Queries are processed simultaneously:
                #
                #   Query A attends to:
                #       [1 2 3 4 5]
                #
                #   Query B attends to:
                #       [1 2 3 4 5 A]
                #
                #   Query C attends to:
                #       [1 2 3 4 5 A B]
                #
                # The last token in the chunk requires the largest attention range:
                #
                #   W old cache tokens
                #   + (chunk_size - 1) earlier chunk tokens
                #
                # Therefore attention computation must temporarily keep:
                #
                #   sliding_window_size + num_tokens - 1
                #
                # keys/values so later tokens in the chunk can attend to earlier chunk tokens.
                attn_keep = min(keys.size(1), self.sliding_window_size + num_tokens - 1)
                keys = keys[:, -attn_keep:, :, :]
                values = values[:, -attn_keep:, :, :]

                cache_keep = min(keys.size(1), self.sliding_window_size)
                self.cache_k = combined_k[:, -cache_keep:, :, :]
                self.cache_v = combined_v[:, -cache_keep:, :, :]

            else:
                self.cache_k, self.cache_v = combined_k, combined_v

            dropped = combined_k.size(1) - keys.size(1)
            k_start_pos_abs = (self.ptr_current_pos - old_len) + dropped
            q_start_pos_abs = self.ptr_current_pos
        else:
            keys, values = keys_new, values_new


        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # causal + sliding-window mask
        num_tokens_Q = queries.shape[-2]
        num_tokens_K = keys.shape[-2]
        device = queries.device

        if use_cache:
            q_start = q_start_pos_abs
            k_start = k_start_pos_abs
        else:
            q_start = 0
            k_start = 0

        q_positions = torch.arange(q_start, q_start + num_tokens_Q, device=device, dtype=torch.long)
        k_positions = torch.arange(k_start, k_start + num_tokens_K, device=device, dtype=torch.long)
        
        # Sliding window width
        W = num_tokens_K + 1 if self.sliding_window_size is None else int(self.sliding_window_size)
        diff = q_positions.unsqueeze(-1) - k_positions.unsqueeze(0)
        mask_bool = (diff < 0) | (diff >= W)

        if use_cache:
            self.ptr_current_pos += num_tokens_Q
        else:
            self.ptr_current_pos = 0

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec
    
    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.ptr_current_pos = 0


import torch
import torch.nn as nn

swa_attn = MultiHeadAttentionWithSWA(
    d_in=768,
    d_out=768,
    dropout=0.1, 
    num_heads=12, 
    qkv_bias=False,
    sliding_window_size=256  # Only looks back at the last 256 tokens
)

batch_size = 2
seq_len = 1024

out = swa_attn(embeddings, use_cache=True)

print("Output shape:", out.shape)

Output shape: torch.Size([8, 1024, 768])


## Gated Attention

In [43]:
import torch
from torch import nn

class GatedMultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_gate = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
            persistent=False
        )

    def forward(self, x):
        b, num_tokens, _ = x.shape
        queries = self.W_query(x)
        gate = self.W_gate(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1,2)
        values = values.transpose(1,2)
        queries = queries.transpose(1,2)

        attn_scores = queries @ keys.transpose(2,3)
        attn_scores.masked_fill_(self.mask.bool()[ :num_tokens, :num_tokens], -torch.inf)

        attn_weights = torch.softmax(attn_scores / (self.head_dim ** 0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = (attn_weights @ values).transpose(1,2)
        context = context.reshape(b, num_tokens, self.d_out)

        context = context * torch.sigmoid(gate)

        out = self.out_proj(context)
        
        return out



gated_attn = GatedMultiHeadAttention(
    d_in=256,
    d_out=256,
    dropout=0.1, 
    num_heads=4, 
    context_length=context_length,
    qkv_bias=False,
)


out = gated_attn(input_embeddings)

print("Output shape:", out.shape)



Output shape: torch.Size([8, 4, 256])
